# 02 · One catalog-inclusive music graph
Union catalog and playlist tracks by Spotify ID. Retain every track–artist credit. Add artist–genre edges and optional sourced artist–country edges. Shared track credits provide collaboration paths. Split playlist edges before any graph propagation.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/data_pipeline.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
import pandas as pd
from scripts.data_pipeline import OUT, REPORTS


In [2]:
from scripts.data_pipeline import integrate
coverage = integrate()

{
  "tracks": 1309344,
  "artists": 1199540,
  "playlists": 18829,
  "metadata_nodes": 5366,
  "total_nodes": 2533079,
  "catalog_only_tracks": 575407,
  "playlist_only_tracks": 403855,
  "overlapping_tracks": 11265,
  "track_artist_edges": 1480427,
  "split_counts": {
    "train": 734897,
    "val": 144907,
    "test": 144907
  },
  "countries_available": 0,
  "elvana_gjata_artist_records": 1,
  "artists_without_track_records": 1022778,
  "elvana_gjata_tracks": 0,
  "elvana_gjata_playlist_tracks": 0,
  "graph_uses_only_train_playlist_edges": true
}


For each eligible playlist, hide 15% of edges for validation and 15% for test (with at least one in each). All other edges are training data. Negative sampling excludes every known positive. Static artist/audio metadata is available for the full catalog, but validation/test playlist composition never enters graph features.

In [3]:
display(pd.read_parquet(OUT / "tracks.parquet").head(10))
display(pd.read_parquet(OUT / "track_artist_edges.parquet").head(10))
display(pd.read_parquet(OUT / "split.parquet").groupby("split").size())

,spotify_track_id,track_title,artist_name,track_node_id,in_catalog,in_playlists,spud_popularity,audio_features_available
0,0000QuApNltQzqS5ROXcQ7,Memories Are Made of This,Dean Martin,0,False,True,0.00,False
1,0000korRHja9p9XaR5UA5m,Girls Just Want To Have Fun,Cyndi Lauper,1,False,True,0.21,False
2,000116ufvkkgMsZB0cOjHK,Dream Of The Dolphins Part 2,Leviathan,2,False,True,0.00,False
3,0001JarZSrH1DwJruLt8YK,Bonaparte's Retreat,Kay Starr,3,False,True,0.00,False
4,0001KlxKxXdIbexJkKRSDL,Uh-Oh (Dj Rupture's Coma Teen Chart Damage Remix),Nice Nice,4,False,False,0.00,False
5,0002buwAM3UowoloZ0APTA,I Hear You Knocking,Smiley Lewis,5,False,True,0.02,False
6,0002iqtszIMQllda15bG4i,On A Hill In A Bed On A Road In A House,High Places,6,False,True,0.00,False
7,0004Uy71ku11n3LMpuyf59,24.11.94 - Wersja Akustyczna,Golden Life,7,True,False,0.00,True
8,00063WYN7QLWhLWn2rScoz,Laura,Les Brown,8,False,True,0.00,False
9,00069XALwNJ1CyknR5z8Vb,Delincuencia,La Polla Records,9,False,True,0.00,False


,spotify_track_id,spotify_artist_id,track_node_id,artist_node_id
0,35iwgR4jXetI318WEWsa1Q,45tIt06XoI0Iio4LBEVpls,676822,654948
1,021ht4sdgPcrDgSk7JTbKY,14jtPCOoNZwquk5wd9DxrY,14300,206219
2,07A5yehtSnoedViJAZkNnc,5LiOoJbxVSAMkBS2fUm3X2,44903,840305
3,08FmqUhxtyLTn6pAh6bk45,5LiOoJbxVSAMkBS2fUm3X2,50958,840305
4,08y9GfoqCWfOGsKdwojr5e,3BiJGZsyX9sJchTqcSA7Su,54907,520934
5,0BRXJHRNGQ3W4v9frnSfhu,3BiJGZsyX9sJchTqcSA7Su,67929,520934
6,0Dd9ImXtAtGwsmsAD69KZT,2nuMRGzeJ5jJEKlfS7rZ0W,78936,464232
7,0IA0Hju8CAgYfV1hwhidBH,4AxgXfD7ISvJSTObqm4aIE,100916,667008
8,0IgI1UCz84pYeVetnl1lGP,5nWlsH5RDgFuRAiDeOFVmf,103323,906357
9,0JV4iqw2lSKJaHBQZ0e5zK,5LiOoJbxVSAMkBS2fUm3X2,107278,840305


split
test     144907
train    734897
val      144907
dtype: int64

Compute normalized sparse propagation once, writing each hop to a memory-mapped file. This is algebraically equivalent to linear LightGCN propagation of projected features and supports the SIGN encoders without retaining full-graph gradients in RAM.

In [4]:
from scripts.data_pipeline import propagate
propagate(max_hops=4)

Graph propagation 1/4 saved


Graph propagation 2/4 saved


Graph propagation 3/4 saved


Graph propagation 4/4 saved
